# 42. Larger datasets and incremental learning
A table that does not fit in memory requires a different workflow. Some estimators support partial_fit, updating a model batch by batch. This is not the same as fit repeatedly: ordinary fit usually starts a new learning process. Incremental learning also needs consistent features, suitable scaling, and an evaluation stream that avoids future-label leakage.

Use sparse storage for sparse features. HashingVectorizer creates a fixed feature space without learning a vocabulary but may have collisions. SGD estimators support incremental updates. For classification, provide all possible classes on the first partial_fit call. Track memory, batch order, and held-out performance. Cache only where repeated transformations are deterministic and beneficial.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
X, y = make_classification(n_samples=200, n_features=8, random_state=42) # Small stand-in for a stream.
model = SGDClassifier(loss="log_loss", random_state=42) # An incremental classifier.
for start in range(0, 160, 40): # Train on four batches and reserve the last forty rows.
    model.partial_fit(X[start:start+40], y[start:start+40], classes=np.array([0, 1])) # Keep feature and class contracts fixed.
print("Held-out accuracy:", round(model.score(X[160:], y[160:]), 3)) # Evaluate unseen rows.
print("Seen training rows:", 160) # Record the training boundary.
assert model.n_features_in_ == 8 # Check consistent feature width.

## Your turn
Compare two batch sizes and record elapsed time, memory, and held-out results. Explain why labels from the future cannot update an earlier evaluation snapshot.

Try the task before opening the solution on the course website.